In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb
import pickle
import warnings
import datetime
import pandas_datareader as pdr
import os
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.api import VAR
from sklearn.linear_model import LinearRegression

from pathlib import Path
os.chdir(Path().resolve().parent)

import warnings
warnings.filterwarnings('ignore')

In [3]:
# загружаем макроданные
start = datetime.datetime(2007, 1, 1)
end   = datetime.datetime(2018, 12, 31)

fed_rate     = pdr.get_data_fred('FEDFUNDS',  start, end)
unemployment = pdr.get_data_fred('UNRATE',    start, end)
inflation    = pdr.get_data_fred('CPIAUCSL',  start, end)

inflation['CPI_YOY'] = inflation['CPIAUCSL'].pct_change(12) * 100

macro = pd.DataFrame({
    'fed_rate':     fed_rate['FEDFUNDS'],
    'unemployment': unemployment['UNRATE'],
    'inflation':    inflation['CPI_YOY']
}).dropna()


In [4]:
# ADF тест на стационарность
def adf_test(series, name):
    result = adfuller(series, autolag='AIC')
    print(f"{name}:")
    print(f"  ADF статистика: {result[0]:.4f}")
    print(f"  p-value:        {result[1]:.4f}")
    print(f"  Вывод: {'Стационарен' if result[1] < 0.05 else 'НЕ стационарен'}")
    print()

print("=== Тест на стационарность (уровни) ===")
for col in ['fed_rate', 'unemployment', 'inflation']:
    adf_test(macro[col], col)

=== Тест на стационарность (уровни) ===
fed_rate:
  ADF статистика: 2.6463
  p-value:        0.9991
  Вывод: НЕ стационарен

unemployment:
  ADF статистика: -0.9584
  p-value:        0.7681
  Вывод: НЕ стационарен

inflation:
  ADF статистика: -2.0257
  p-value:        0.2754
  Вывод: НЕ стационарен



In [5]:
macro['d_fed_rate']     = macro['fed_rate'].diff()
macro['dl_unemployment'] = np.log(macro['unemployment']).diff()
macro['d_inflation']    = macro['inflation'].diff()

macro_diff = macro[['d_fed_rate', 'dl_unemployment', 'd_inflation']].dropna()

print(f"Наблюдений после трансформации: {len(macro_diff)}")
print()
print("=== Тест на стационарность (первые разности) ===")
for col in macro_diff.columns:
    adf_test(macro_diff[col], col)

Наблюдений после трансформации: 131

=== Тест на стационарность (первые разности) ===
d_fed_rate:
  ADF статистика: -7.7088
  p-value:        0.0000
  Вывод: Стационарен

dl_unemployment:
  ADF статистика: -4.1959
  p-value:        0.0007
  Вывод: Стационарен

d_inflation:
  ADF статистика: -4.6999
  p-value:        0.0001
  Вывод: Стационарен



In [6]:
# эндогенные переменные
endog = macro_diff[['dl_unemployment', 'd_inflation']]

# экзогенная — ставка ФРС
exog = macro_diff[['d_fed_rate']]

# подбор оптимального числа лагов
model = VAR(endog, exog=exog)
lag_selection = model.select_order(maxlags=6)
print(lag_selection.summary())

 VAR Order Selection (* highlights the minimums) 
      AIC         BIC         FPE         HQIC   
-------------------------------------------------
0      -9.047      -8.956   0.0001178      -9.010
1      -9.268     -9.087*   9.439e-05     -9.195*
2      -9.302      -9.030   9.129e-05      -9.191
3      -9.268      -8.906   9.446e-05      -9.121
4     -9.303*      -8.851  9.120e-05*      -9.119
5      -9.293      -8.750   9.217e-05      -9.072
6      -9.277      -8.643   9.375e-05      -9.019
-------------------------------------------------


In [7]:
model_varx = VAR(endog, exog=exog)
results_varx = model_varx.fit(maxlags=1)
print(results_varx.summary())

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 21, Apr, 2026
Time:                     23:19:08
--------------------------------------------------------------------
No. of Equations:         2.00000    BIC:                   -8.98513
Nobs:                     130.000    HQIC:                  -9.08989
Log likelihood:           234.580    FPE:                0.000105000
AIC:                     -9.16159    Det(Omega_mle):     9.88245e-05
--------------------------------------------------------------------
Results for equation dl_unemployment
                        coefficient       std. error           t-stat            prob
-------------------------------------------------------------------------------------
const                     -0.002158         0.002342           -0.921           0.357
d_fed_rate                -0.066614         0.020903           -3.187           0.001
L1.dl_unemployment         0.0

In [9]:
con = duckdb.connect('notebooks/credit_risk.db')

with open('models/model_lr_v2.pkl', 'rb') as f:
    model_lr = pickle.load(f)
with open('models/scaler_v2.pkl', 'rb') as f:
    scaler = pickle.load(f)

df = con.execute("SELECT * FROM mart.features").df()

FEATURES = [
    'dti', 'fico_avg', 'annual_inc', 'loan_amnt',
    'installment', 'open_acc', 'revol_util', 'total_acc',
    'loan_to_income', 'payment_to_income',
    'term_months', 'home_ownership', 'purpose'
]

df_model = df[FEATURES + ['is_default', 'issue_date']].dropna()
df_model = pd.get_dummies(df_model,
                           columns=['home_ownership', 'purpose'],
                           drop_first=True)
FEATURES_ENC = [c for c in df_model.columns 
                if c not in ['is_default', 'issue_date']]

df_model['pd_pred'] = model_lr.predict_proba(
    scaler.transform(df_model[FEATURES_ENC])
)[:, 1]

# агрегируем по месяцам
df_model['month'] = pd.to_datetime(df_model['issue_date']).dt.to_period('M')
monthly = df_model.groupby('month').agg(
    pd_mean=('pd_pred', 'mean'),
    default_rate=('is_default', 'mean'),
    n=('is_default', 'count')
).reset_index()

monthly['month'] = monthly['month'].dt.to_timestamp()
monthly = monthly[monthly['n'] > 100]  # убираем месяцы с малым числом кредитов

print(f"Месяцев: {len(monthly)}")
print(monthly.head())

Месяцев: 126
        month   pd_mean  default_rate    n
7  2008-01-01  0.132810      0.182353  170
8  2008-02-01  0.134659      0.143678  174
9  2008-03-01  0.138440      0.165217  230
10 2008-04-01  0.091173      0.174194  155
17 2008-11-01  0.128794      0.169399  183


In [10]:
# объединяем с макроданными
monthly_macro = monthly.merge(
    macro[['fed_rate', 'unemployment', 'inflation']].reset_index(),
    left_on='month', right_on='DATE', how='inner'
)

print(f"Совместных наблюдений: {len(monthly_macro)}")

# регрессия: default_rate ~ fed_rate + unemployment + inflation
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm

X_macro = monthly_macro[['fed_rate', 'unemployment', 'inflation']]
y_macro  = monthly_macro['default_rate']

# используем statsmodels для p-value
X_macro_sm = sm.add_constant(X_macro)
ols = sm.OLS(y_macro, X_macro_sm).fit()
print(ols.summary())

Совместных наблюдений: 126
                            OLS Regression Results                            
Dep. Variable:           default_rate   R-squared:                       0.473
Model:                            OLS   Adj. R-squared:                  0.460
Method:                 Least Squares   F-statistic:                     36.46
Date:                Tue, 21 Apr 2026   Prob (F-statistic):           6.81e-17
Time:                        23:20:24   Log-Likelihood:                 246.12
No. Observations:                 126   AIC:                            -484.2
Df Residuals:                     122   BIC:                            -472.9
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.31

In [11]:
monthly_macro['default_rate_lag1'] = monthly_macro['default_rate'].shift(1)
monthly_macro['default_rate_lag2'] = monthly_macro['default_rate'].shift(2)
monthly_macro_adl = monthly_macro.dropna()

X_final = monthly_macro_adl[[
    'default_rate_lag1',
    'fed_rate'
]]

X_final_sm = sm.add_constant(X_final)
ols_final = sm.OLS(monthly_macro_adl['default_rate'], X_final_sm).fit()

dw = sm.stats.stattools.durbin_watson(ols_final.resid)
print(f"R²:            {ols_final.rsquared:.3f}")
print(f"Adj R²:        {ols_final.rsquared_adj:.3f}")
print(f"Durbin-Watson: {dw:.3f}")
print(ols_final.summary())

R²:            0.890
Adj R²:        0.889
Durbin-Watson: 2.070
                            OLS Regression Results                            
Dep. Variable:           default_rate   R-squared:                       0.890
Model:                            OLS   Adj. R-squared:                  0.889
Method:                 Least Squares   F-statistic:                     491.6
Date:                Tue, 21 Apr 2026   Prob (F-statistic):           8.06e-59
Time:                        23:23:29   Log-Likelihood:                 338.80
No. Observations:                 124   AIC:                            -671.6
Df Residuals:                     121   BIC:                            -663.1
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------

In [14]:
# последнее наблюдение из данных — отправная точка
last_default_rate = monthly_macro_adl['default_rate'].iloc[-1]
last_fed_rate     = monthly_macro_adl['fed_rate'].iloc[-1]

print(f"Последний default_rate: {last_default_rate:.4f}")
print(f"Последняя ставка ФРС:   {last_fed_rate:.2f}%")

# коэффициенты из модели
const    = ols_final.params['const']
beta_lag = ols_final.params['default_rate_lag1']
beta_fed = ols_final.params['fed_rate']

# ADL модель показала контринтуитивный знак при ставке ФРС
# из-за специфики данных Lending Club (рост платформы в период низких ставок)
# поэтому используем прямой сценарный анализ по PD

base_pd   = monthly_macro_adl['default_rate'].iloc[-1]
total_ead = con.execute("SELECT SUM(ead) FROM mart.features").fetchone()[0]

print(f"Базовый PD портфеля: {base_pd:.4f}")
print(f"Общий EAD:           {total_ead:,.0f}")

Последний default_rate: 0.0106
Последняя ставка ФРС:   2.27%
Базовый PD портфеля: 0.0106
Общий EAD:           19,305,067,850


In [13]:
np.random.seed(42)
rho    = 0.15
N_SIM  = 10_000
CONF   = 0.99

df_port = df[FEATURES + ['ead']].dropna()
df_port = pd.get_dummies(df_port,
                          columns=['home_ownership', 'purpose'],
                          drop_first=True)
FEATURES_ENC = [c for c in df_port.columns if c != 'ead']

pd_base = model_lr.predict_proba(
    scaler.transform(df_port[FEATURES_ENC])
)[:, 1]

lgds = np.full(len(df_port), 0.697)
eads = df_port['ead'].values

def compute_var_asrf(pds, lgds, eads, rho, n_sim, conf):
    K      = scipy_stats.norm.ppf(np.clip(pds, 0.001, 0.999))
    losses = np.zeros(n_sim)
    for i in range(n_sim):
        Z        = np.random.normal(0, 1)
        epsilon  = np.random.normal(0, 1, len(pds))
        Y        = np.sqrt(rho) * Z + np.sqrt(1 - rho) * epsilon
        defaults = (Y < K).astype(int)
        losses[i] = np.sum(defaults * lgds * eads)
    el  = np.mean(losses)
    var = np.percentile(losses, conf * 100)
    es  = losses[losses >= var].mean()
    return el, var, es

results_stress = {}

for name, stressed_dr in scenarios.items():
    scale        = stressed_dr / base_pd
    pd_stressed  = np.clip(pd_base * scale, 0.001, 0.999)
    
    el, var, es = compute_var_asrf(pd_stressed, lgds, eads, rho, N_SIM, CONF)
    
    results_stress[name] = {
        'PD_mean':   round(pd_stressed.mean(), 4),
        'EL':        el,
        'VaR_99':    var,
        'ES_99':     es,
        'EL_ratio':  el  / eads.sum(),
        'VaR_ratio': var / eads.sum()
    }
    print(f"\n{name}")
    print(f"  Средний PD:  {pd_stressed.mean():.4f}")
    print(f"  EL:          {el/1e9:.3f}B")
    print(f"  VaR 99%:     {var/1e9:.3f}B")
    print(f"  VaR / EAD:   {var/eads.sum():.3f}")


Базовый PD (ставка ФРС ~2.3% (факт 2018))
  Средний PD:  0.1996
  EL:          2.898B
  VaR 99%:     7.006B
  VaR / EAD:   0.363

Стресс 1 — умеренный, +50% (ставка ФРС ~4-5%)
  Средний PD:  0.2993
  EL:          4.345B
  VaR 99%:     8.572B
  VaR / EAD:   0.444

Стресс 2 — тяжёлый, +150%, (ставка ФРС ~6-7%)
  Средний PD:  0.4836
  EL:          6.962B
  VaR 99%:     10.597B
  VaR / EAD:   0.549

Стресс 3 — кризис, +400% (ставка ФРС >8%)
  Средний PD:  0.7746
  EL:          10.733B
  VaR 99%:     12.500B
  VaR / EAD:   0.648


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names       = list(results_stress.keys())
var_values  = [results_stress[n]['VaR_99'] / 1e9 for n in names]
el_values   = [results_stress[n]['EL']     / 1e9 for n in names]
x           = np.arange(len(names))
width       = 0.35

# левый график — VaR и EL рядом
axes[0].bar(x - width/2, var_values, width, label='VaR 99%', color='steelblue', alpha=0.8)
axes[0].bar(x + width/2, el_values,  width, label='EL',      color='green',     alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=15, ha='right')
axes[0].set_title('VaR 99% и EL по сценариям')
axes[0].set_ylabel('Потери (млрд)')
axes[0].legend()

# правый график — PD по сценариям
pd_values = [results_stress[n]['PD_mean'] for n in names]
axes[1].bar(x, pd_values, color='steelblue', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=15, ha='right')
axes[1].set_title('Средний PD по сценариям')
axes[1].set_ylabel('PD')

plt.tight_layout()
plt.show()


In [16]:
con.close()